In [1]:
#SIMULATION - POLICY INTERVENTION (Readme)
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Module conducts the following analysis:

# ---- Policy intervention simulation

# Module is input for:

# ---- Simulation-policy-effectiveness#
# ---- Simulation-treatment

In [2]:
# set-up
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# set working directory

# function to install and load required packages
check_and_load <- function(packages) {
  for (pkg in packages) {
    # install if missing
    if (!requireNamespace(pkg, quietly = TRUE)) {
      message(paste("Installing missing package:", pkg))
      install.packages(pkg, dependencies = TRUE, repos = "https://cloud.r-project.org")
    }
    # load if not already attached
    if (!(pkg %in% (.packages()))) {
      suppressPackageStartupMessages(library(pkg, character.only = TRUE))
    }
  }
}

# required libraries
required_packages <- c(
  "dplyr",      # data manipulation (mutate, group_by, summarise)
  "Matrix",     # sparse matrices (spatial weights)
  "FNN",        # k-nearest neighbors
  "sf",         # spatial data handling
  "tidyr",      # reshaping and NA handling
  "tibble",     # tibble structures
  "purrr",      # functional programming (map, etc.)
  "data.table", # fast aggregation and joins
  "ggplot2",    # plotting
  "patchwork",   # combining plots
  "ggbreak"    # break axes for visualization
)

# load all required packages
check_and_load(required_packages)

ERROR: Error in c("dplyr", "Matrix", "FNN", "sf", "tidyr", "tibble", "purrr", : argument 12 is empty


In [ ]:
#INPUTS AND SETTINGS
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# --- Calibration Parameters ---
distance_cutoff_rotterdam <- 50000           # [m] Max distance for seeding initial adopters around Rotterdam
rotterdam_coords <- cbind(4.4786, 51.9244)   # Rotterdam coordinates (reference point for initialization)
start_year <- 1961                           # Start year of diffusion in calibration set
noise_multiplier <- 0.5                      # Noise multiplier for second-pass calibration (to avoid overfitting)
k_max_number <- 1000                          # Max number of neighbors for KNN spatial weights
km_cutoff <- 120000                          # [m] Distance cutoff for adjacency matrix (optimized in calibration)

# --- Simulation Parameters ---
initial_share <- 0.01                        # Share of adopters at start of simulation (as fraction of eligible offtakers)
carbon_price_setting <- "carbon_price"       # Fossil scenario: "carbon_price", "flat_carbon_price", or "no_carbon_price"
n_simulations <- 250                         # Number of Monte Carlo simulation runs
start_year_sim <- 2024                       # Start year for prospective simulation
end_year <- 2100                             # End year for prospective simulation
intervention_volume <- 0.1                   # Fraction of plants targeted per sector-quantile group
quantile <- 4                                # Sectoral stratification granularity (quartiles)
seeds <- 1:n_simulations                     # reproducibility
saturation_setting <- "central"              # switch to "restricted" / "extended" if needed
cost_setting <- "mean"                       # switch to conservative / progressive if needed
simulation_years <- start_year_sim:end_year  # simulation years

In [ ]:
# DATA FILES
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
offtakers <- readRDS("/home/h1604190/First Submission/offtakers_centrality.rds")
cost_gap_data <- readRDS("/home/h1604190/Data/cost_gap_data.rds")
spatial_weights <- readRDS("/home/h1604190/First Submission/spatial_weights.rds")

In [ ]:
# helper functions
summarize_run <- function(run_result) {
  run_result$adoption_plant %>%
    dplyr::left_join(run_result$policy_cost, by = c("year", "plant_id")) %>%
    dplyr::group_by(year) %>%
    dplyr::summarise(
      annual_demand = sum(adoption * hydrogen_2050, na.rm = TRUE),
      annual_cost = sum(policy_cost, na.rm = TRUE),
      .groups = "drop"
    )
}

adopt_prob <- function(beta0, beta1, spatial_influence_detrended,
                       beta2, cost_diff,
                       beta3, distance_to_waterway,
                       beta4, distance_to_pipeline,
                       beta5) {
  1 / (1 + exp(-(beta0 +
    beta1 * spatial_influence_detrended +
    beta2 * cost_diff +
    beta3 * distance_to_waterway +
    beta4 * distance_to_pipeline +
    beta5 * spatial_influence_detrended * cost_diff)))
}

compute_cost_need <- function(years, sector_chr, cost_diff_lookup) {
  out <- numeric(length(sector_chr))

  for (yy in years) {
    cost_year <- cost_diff_lookup[[as.character(yy)]]

    if (is.null(cost_year) || !nrow(cost_year)) next

    cost_vec <- cost_year$cost_diff
    names(cost_vec) <- as.character(cost_year$sector)

    cd <- unname(cost_vec[sector_chr])
    cd[is.na(cd)] <- 0

    out <- out + pmin(cd, 0)
  }

  -out
}

percent_rank_fast <- function(x) {
  x2 <- x
  ok <- !is.na(x2)
  out <- rep(NA_real_, length(x2))

  if (!any(ok)) return(out)

  r <- rank(x2[ok], ties.method = "average")
  n <- sum(ok)

  if (n == 1) {
    out[ok] <- 0
  } else {
    out[ok] <- (r - 1) / (n - 1)
  }

  out
}

# global settings
cost_weight_global <- 0.9
cent_weight_global <- 1 - cost_weight_global

cost_weight_grid <- seq(0.5, 1, by = 0.1)
cost_weight_grid <- round(cost_weight_grid, 1)

plot_theme_example <- theme_minimal(base_size = 20) +
  theme(
    panel.grid = element_blank(),
    panel.border = element_rect(color = "black", fill = NA),
    axis.line = element_line(color = "black"),
    axis.text = element_text(size = 18),
    strip.text = element_text(size = 18, face = "plain"),
    strip.background = element_blank(),
    legend.title = element_blank()
  )

cols_contrast <- c(
  "diff_HvsC" = "#2C7FB8"
)

# offtaker data
offtakers_df <- offtakers %>%
  sf::st_set_geometry(NULL) %>%
  mutate(beta6 = get(paste0("beta6_", saturation_setting))) %>%
  select(
    plant_id, sector, hydrogen_2050,
    beta0, beta1, beta2, beta3, beta4, beta5, beta6,
    distance_to_rotterdam, distance_to_waterway, distance_to_pipeline,
    betweenness, degree
  )

# core vectors
sector <- offtakers_df$sector
plant_id <- offtakers_df$plant_id
n_offtakers <- nrow(offtakers_df)
unique_sectors <- unique(sector)

beta0_vec <- offtakers_df$beta0
beta1_vec <- offtakers_df$beta1
beta2_vec <- offtakers_df$beta2
beta3_vec <- offtakers_df$beta3
beta4_vec <- offtakers_df$beta4
beta5_vec <- offtakers_df$beta5
beta6_vec <- offtakers_df$beta6

distance_rotterdam <- offtakers_df$distance_to_rotterdam
distance_waterway <- offtakers_df$distance_to_waterway
distance_pipeline <- offtakers_df$distance_to_pipeline
hydrogen_2050 <- offtakers_df$hydrogen_2050

# sector structure
sector_index <- split(seq_len(n_offtakers), sector)
sector_factor <- as.integer(factor(sector, levels = unique_sectors))
n_sectors <- length(unique_sectors)
sector_counts <- tabulate(sector_factor, nbins = n_sectors)

# cost lookup
cost_diff_lookup <- cost_gap_data %>%
  filter(
    green_scenario == cost_setting,
    fossil_scenario == carbon_price_setting
  ) %>%
  select(year, sector, cost_diff) %>%
  distinct(year, sector, .keep_all = TRUE) %>%
  mutate(sector = as.character(sector)) %>%
  split(.$year)

# intervention setup
centrality_types <- list(
  betweenness = offtakers_df$betweenness,
  degree = offtakers_df$degree
)

intervention_types <- c("baseline", "high", "random")

# initial adopters
eligible_init <- which(
  distance_rotterdam <= distance_cutoff_rotterdam &
    beta6_vec > 0
)

set.seed(42)

selected_indices <- if (length(eligible_init) > 0) {
  sample(
    eligible_init,
    size = min(length(eligible_init), max(1, round(initial_share * length(eligible_init)))),
    replace = FALSE
  )
} else {
  integer(0)
}

initial_adoption <- rep.int(0, n_offtakers)
initial_adoption[selected_indices] <- 1

simulation_years <- start_year_sim:end_year

# simulation function
run_simulation <- function(intervention_type = "baseline",
                           seed = 19,
                           contribution_score,
                           cost_weight = cost_weight_global,
                           cent_weight = cent_weight_global) {

  set.seed(seed)

  # time setup
  years <- simulation_years
  n_years <- length(years)

  # storage
  adoption_mat <- matrix(0, n_offtakers, n_years)
  cost_diff_mat <- matrix(0, n_offtakers, n_years)
  original_cost_diff_mat <- matrix(0, n_offtakers, n_years)
  policy_cost_mat <- matrix(0, n_offtakers, n_years)

  influence_summary_list <- vector("list", n_years)
  influence_plant_list <- vector("list", n_years)

  target_offtakers <- integer(0)

  # targeting scores
  cost_need <- compute_cost_need(years, as.character(sector), cost_diff_lookup)
  cost_q <- 1 - percent_rank_fast(cost_need)
  cent_q <- percent_rank_fast(contribution_score)

  score_df <- tibble(
    idx = seq_len(n_offtakers),
    plant_id = plant_id,
    sector = sector,
    cost_q = cost_q,
    cent_q = cent_q
  )

  jitter <- runif(n_offtakers, 0, 1e-9)

  # target selection
  if (intervention_type != "baseline") {
    k <- max(1L, ceiling(intervention_volume * n_offtakers))

    cent_component <- switch(
      intervention_type,
      high = cent_q,
      random = 0,
      0
    )

    if (intervention_type == "random") {
      cw <- 1
      sw <- 0
    } else {
      cw <- cost_weight
      sw <- cent_weight
    }

    composite_score <- cw * cost_q + sw * cent_component
    composite_score[is.na(composite_score)] <- -Inf

    ord <- order(composite_score + jitter, decreasing = TRUE)

    target_offtakers <- ord[seq_len(min(k, length(ord)))]
    target_offtakers <- target_offtakers[is.finite(composite_score[target_offtakers])]

    score_df <- score_df %>%
      mutate(
        composite_score = composite_score,
        treated = idx %in% target_offtakers,
        intervention_type = intervention_type
      )
  } else {
    score_df <- score_df %>%
      mutate(
        composite_score = NA_real_,
        treated = FALSE,
        intervention_type = intervention_type
      )
  }

  treated_flag_vec <- logical(n_offtakers)
  if (length(target_offtakers)) treated_flag_vec[target_offtakers] <- TRUE

  previous_adoption <- initial_adoption

  # sector adoption types
  continuous_sectors <- c("Heavy duty", "Aviation", "Shipping")
  continuous_flag <- sector %in% continuous_sectors
  discrete_sectors <- unique_sectors[!unique_sectors %in% continuous_sectors]

  # simulation loop
  for (tt in seq_len(n_years)) {
    year <- years[tt]

    # spatial influence
    spatial_influence <- as.numeric(spatial_weights %*% previous_adoption)
    spatial_influence[is.na(spatial_influence)] <- 0
    spatial_influence_detrended <- spatial_influence - mean(spatial_influence)

    # influence outputs
    if (length(target_offtakers)) {
      influence_summary_list[[tt]] <- tibble(
        year = year,
        treated_influence_received = sum(spatial_influence[treated_flag_vec]),
        untreated_influence_received = sum(spatial_influence[!treated_flag_vec]),
        mean_adoption_treated = mean(previous_adoption[treated_flag_vec]),
        mean_adoption_untreated = mean(previous_adoption[!treated_flag_vec])
      )

      influence_plant_list[[tt]] <- tibble(
        year = year,
        plant_id = plant_id,
        spatial_influence = spatial_influence,
        treated = treated_flag_vec
      )
    }

    # annual cost diff
    cost_year <- cost_diff_lookup[[as.character(year)]]

    if (is.null(cost_year)) {
      cost_diff <- rep(0, n_offtakers)
    } else {
      cost_vec <- cost_year$cost_diff
      names(cost_vec) <- cost_year$sector
      cost_diff <- unname(cost_vec[sector])
      cost_diff[is.na(cost_diff)] <- 0
    }

    original_cost_diff <- cost_diff

    # intervention effect
    if (length(target_offtakers)) {
      cost_diff[target_offtakers] <- pmax(0, cost_diff[target_offtakers])
    }

    cost_diff_mat[, tt] <- cost_diff
    original_cost_diff_mat[, tt] <- original_cost_diff

    # raw adoption probability
    p_raw <- adopt_prob(
      beta0_vec,
      beta1_vec, spatial_influence_detrended,
      beta2_vec, cost_diff,
      beta3_vec, distance_waterway,
      beta4_vec, distance_pipeline,
      beta5_vec
    )
    p_raw[is.na(p_raw)] <- 0

    # residual adoption space
    sector_sum <- rowsum(previous_adoption, group = sector_factor, reorder = FALSE)
    sector_mean <- as.numeric(sector_sum) / sector_counts
    sector_mean_vec <- sector_mean[sector_factor]

    residual_share <- pmax(
      0,
      (beta6_vec - sector_mean_vec) / pmax(1 - sector_mean_vec, 1e-6)
    )

    residual_p <- p_raw * residual_share
    residual_p[is.na(residual_p)] <- 0

    # adoption update
    adoption <- previous_adoption

    if (any(continuous_flag)) {
      idx_c <- which(continuous_flag)
      adoption[idx_c] <- 1 - (1 - adoption[idx_c]) * (1 - residual_p[idx_c])
    }

    for (s in discrete_sectors) {
      idx_s <- sector_index[[s]]
      adoption[idx_s] <- pmax(
        adoption[idx_s],
        rbinom(length(idx_s), 1, residual_p[idx_s])
      )
    }

    adoption[beta6_vec == 0] <- 0
    adoption_mat[, tt] <- adoption

    # policy cost
    policy_cost_mat[, tt] <- ifelse(
      original_cost_diff < 0 & cost_diff == 0,
      hydrogen_2050 * (-original_cost_diff) * adoption * 33000 / 1e9,
      0
    )

    previous_adoption <- adoption
  }

  # long outputs
  year_rep <- rep(years, each = n_offtakers)
  plant_rep <- rep(plant_id, times = n_years)
  sector_rep <- rep(sector, times = n_years)
  h2050_rep <- rep(hydrogen_2050, times = n_years)

  adoption_plant_df <- tibble(
    year = year_rep,
    plant_id = plant_rep,
    sector = sector_rep,
    adoption = as.vector(adoption_mat),
    cost_diff = as.vector(cost_diff_mat),
    original_cost_diff = as.vector(original_cost_diff_mat),
    hydrogen_2050 = h2050_rep
  )

  policy_cost_df <- tibble(
    year = year_rep,
    plant_id = plant_rep,
    policy_cost = as.vector(policy_cost_mat)
  )

  cost_diff_df <- tibble(
    year = year_rep,
    plant_id = plant_rep,
    cost_diff = as.vector(cost_diff_mat)
  )

  # return
  list(
    adoption_plant = adoption_plant_df,
    policy_cost = policy_cost_df,
    cost_diff = cost_diff_df,
    influence_plant = if (length(target_offtakers)) bind_rows(influence_plant_list) else NULL,
    influence_summary = if (length(target_offtakers)) bind_rows(influence_summary_list) else NULL,
    treated_offtakers = if (length(target_offtakers)) score_df else NULL
  )
}

# run simulations by cost weight
results_all_dist_by_cw <- list()

for (cw in cost_weight_grid) {
  sw <- 1 - cw
  cw_tag <- sprintf("cw%0.1f", cw)

  results_all_dist_cw <- list()

  for (i in seq_len(n_simulations)) {
    seed_i <- seeds[i]

    for (cent_name in names(centrality_types)) {
      score <- centrality_types[[cent_name]]

      for (intervention in intervention_types) {
        key <- paste(cw_tag, cent_name, intervention, sep = "_")

        run_result <- run_simulation(
          intervention_type = intervention,
          seed = seed_i,
          contribution_score = score,
          cost_weight = cw,
          cent_weight = sw
        )

        this_summary <- summarize_run(run_result)
        results_all_dist_cw[[key]][[i]] <- this_summary

        message(sprintf("[%s] finished run %d/%d for %s",
                        Sys.time(), i, n_simulations, key))

        rm(run_result)
        if (i %% 5 == 0) gc()
      }
    }
  }

  results_all_dist_by_cw[[cw_tag]] <- results_all_dist_cw
}

# extract metrics by cost weight
extract_runs_metrics_by_cw <- function(results_all_dist_by_cw) {
  cw_names <- names(results_all_dist_by_cw)
  out_all <- vector("list", length(cw_names))

  for (j in seq_along(cw_names)) {
    cw_tag <- cw_names[[j]]
    results_all_dist_cw <- results_all_dist_by_cw[[cw_tag]]

    nm <- names(results_all_dist_cw)
    out <- vector("list", length(nm))

    for (idx in seq_along(nm)) {
      name <- nm[[idx]]
      runs <- results_all_dist_cw[[name]]

      dt_name <- data.table::rbindlist(
        lapply(seq_along(runs), function(i) {
          s <- runs[[i]]
          data.table::data.table(
            run_id = i,
            year = s$year,
            annual_demand_mt = s$annual_demand / 1e3,
            annual_cost_bn = s$annual_cost,
            scenario_raw = name,
            cost_weight = as.numeric(sub("^cw([0-9.]+)_.*$", "\\1", name))
          )
        })
      )

      out[[idx]] <- dt_name
    }

    DT <- data.table::rbindlist(out)

    # parse scenario labels
    DT[, centrality_type := data.table::fifelse(
      grepl("_betweenness_", scenario_raw), "Betweenness",
      data.table::fifelse(grepl("_degree_", scenario_raw), "Degree", NA_character_)
    )]

    DT[, intervention := data.table::fifelse(
      grepl("_high$", scenario_raw), "High Centrality",
      data.table::fifelse(
        grepl("_random$", scenario_raw), "Cost-only",
        data.table::fifelse(grepl("_baseline$", scenario_raw), "Baseline", NA_character_)
      )
    )]

    DT <- DT[!is.na(centrality_type) & !is.na(intervention)]

    # cumulative metrics
    data.table::setorder(DT, cost_weight, run_id, centrality_type, intervention, year)

    DT[, cumulative_demand_mt := cumsum(annual_demand_mt),
       by = .(cost_weight, run_id, centrality_type, intervention)]

    DT[, cumulative_cost_bn := cumsum(annual_cost_bn),
       by = .(cost_weight, run_id, centrality_type, intervention)]

    # baseline merge
    baseDT <- DT[
      intervention == "Baseline",
      .(
        cost_weight,
        run_id,
        centrality_type,
        year,
        base_demand_mt = cumulative_demand_mt,
        base_cost_bn = cumulative_cost_bn
      )
    ]

    M <- merge(
      DT,
      baseDT,
      by = c("cost_weight", "run_id", "centrality_type", "year"),
      all.x = TRUE
    )

    M[, cumulative_additional_demand_mt := cumulative_demand_mt - base_demand_mt]
    M[, cumulative_additional_cost_bn := cumulative_cost_bn - base_cost_bn]

    # efficiency relative to baseline
    M[, efficiency_eur_per_kg := data.table::fifelse(
      intervention != "Baseline" &
        is.finite(cumulative_additional_demand_mt) &
        is.finite(cumulative_additional_cost_bn) &
        cumulative_additional_demand_mt > 0 &
        cumulative_additional_cost_bn > 0,
      cumulative_additional_cost_bn / cumulative_additional_demand_mt,
      NA_real_
    )]

    out_all[[j]] <- M
  }

  data.table::rbindlist(out_all, use.names = TRUE, fill = TRUE)
}


In [ ]:
make_eff_diff_clip_plot <- function(DT_runs_all, year_target, clip_lim = 1.1) {
  nrc_cols <- ggsci::pal_npg("nrc")(10)
  pos_col  <- nrc_cols[1]
  neg_col  <- nrc_cols[3]
  zero_col <- "black"

  eff_run <- DT_runs_all[
    year == year_target & intervention %in% c("High Centrality", "Cost-only"),
    .(efficiency_eur_per_kg),
    by = .(cost_weight, centrality_type, run_id, intervention)
  ]

  eff_w <- data.table::dcast(
    eff_run,
    cost_weight + centrality_type + run_id ~ intervention,
    value.var = "efficiency_eur_per_kg"
  )

  eff_w[, diff_HvsC := `High Centrality` - `Cost-only`]

  diff_s <- eff_w[, .(
    med = median(pmax(pmin(diff_HvsC, clip_lim), -clip_lim), na.rm = TRUE),
    p25 = quantile(pmax(pmin(diff_HvsC, clip_lim), -clip_lim), 0.25, na.rm = TRUE, type = 1),
    p75 = quantile(pmax(pmin(diff_HvsC, clip_lim), -clip_lim), 0.75, na.rm = TRUE, type = 1)
  ), by = .(cost_weight, centrality_type)]

  diff_s[, sign := data.table::fifelse(
    med > 0, "Positive",
    data.table::fifelse(med < 0, "Negative", "Zero")
  )]

  diff_s[, centrality_type := factor(centrality_type, levels = c("Degree", "Betweenness"))]

  seg_dt <- diff_s[
    order(centrality_type, cost_weight)
  ][
    , .(
      x    = cost_weight,
      xend = data.table::shift(cost_weight, type = "lead"),
      y    = med,
      yend = data.table::shift(med, type = "lead")
    ),
    by = centrality_type
  ][!is.na(xend)]

  seg_dt[, seg_avg := (y + yend) / 2]

  seg_dt[, seg_sign := data.table::fifelse(
    seg_avg > 0, "Positive",
    data.table::fifelse(seg_avg < 0, "Negative", "Zero")
  )]

  ggplot2::ggplot() +
    ggplot2::geom_hline(yintercept = 0, linewidth = 0.6, alpha = 0.6) +
    ggplot2::geom_segment(
      data = seg_dt,
      ggplot2::aes(x = x, xend = xend, y = y, yend = yend, color = seg_sign),
      linewidth = 1.1
    ) +
    ggplot2::geom_errorbar(
      data = diff_s,
      ggplot2::aes(x = cost_weight, ymin = p25, ymax = p75, color = sign),
      width = 0.03,
      linewidth = 0.7,
      alpha = 0.30
    ) +
    ggplot2::geom_point(
      data = diff_s,
      ggplot2::aes(x = cost_weight, y = med, color = sign),
      size = 2.2
    ) +
    ggplot2::facet_grid(. ~ centrality_type, scales = "fixed") +
    ggplot2::scale_color_manual(
      values = c(
        "Positive" = pos_col,
        "Negative" = neg_col,
        "Zero" = zero_col
      )
    ) +
    ggplot2::scale_x_continuous(
      breaks = cost_weight_grid,
      limits = range(cost_weight_grid),
      expand = ggplot2::expansion(mult = c(0.01, 0.05))
    ) +
    ggplot2::scale_y_continuous(limits = c(-clip_lim, clip_lim)) +
    ggplot2::labs(
      title = sprintf("Policy efficiency differences in %d", year_target),
      x = "Cost weight (global)",
      y = "Efficiency difference (EUR kg^-1)"
    ) +
    plot_theme_example +
    ggplot2::theme(
      legend.position = "none",
      plot.title = ggplot2::element_text(size = 20, face = "bold")
    )
}

DT_runs_all <- extract_runs_metrics_by_cw(results_all_dist_by_cw)

p_diff_2035_clip <- make_eff_diff_clip_plot(DT_runs_all, year_target = 2035, clip_lim = 1)
p_diff_2050_clip <- make_eff_diff_clip_plot(DT_runs_all, year_target = 2050, clip_lim = 1)
p_diff_2100_clip <- make_eff_diff_clip_plot(DT_runs_all, year_target = 2100, clip_lim = 1)

panel_clip_years <- p_diff_2035_clip / p_diff_2050_clip / p_diff_2100_clip +
  patchwork::plot_annotation(tag_levels = "a")

options(repr.plot.width = 16, repr.plot.height = 25, repr.plot.res = 600)
print(panel_clip_years)

ggplot2::ggsave(
  "supplementary-figure-15.pdf",
  panel_clip_years,
  device = cairo_pdf,
  width = 16,
  height = 16,
  dpi = 600
)

DT_runs_all <- extract_runs_metrics_by_cw(results_all_dist_by_cw)

p_diff_2035_clip <- make_eff_diff_clip_plot(DT_runs_all, year_target = 2035, clip_lim = 1)
p_diff_2050_clip <- make_eff_diff_clip_plot(DT_runs_all, year_target = 2050, clip_lim = 1)
p_diff_2100_clip <- make_eff_diff_clip_plot(DT_runs_all, year_target = 2100, clip_lim = 1)

panel_clip_years <- p_diff_2035_clip / p_diff_2050_clip / p_diff_2100_clip +
  patchwork::plot_annotation(tag_levels = "a")

options(repr.plot.width = 16, repr.plot.height = 25, repr.plot.res = 600)
print(panel_clip_years)

ggplot2::ggsave(
  "supplementary-figure-15.pdf",
  panel_clip_years,
  device = cairo_pdf,
  width = 16,
  height = 16,
  dpi = 600
)